- pandas
- 讀取data/housing.csv
- 確定沒有空值
- print資料模型 筆數


In [4]:
import pandas as pd
import os

# 1. 確認目前工作目錄
print("目前工作目錄：")
print(os.getcwd())

# 2. 讀取 housing.csv
df = pd.read_csv("../data/housing.csv")

# 3. 確認資料是否有空值
print("\n各欄位空值數量：")
print(df.isnull().sum())

# 確認是否完全沒有空值
if df.isnull().sum().sum() == 0:
    print("\n沒有空值")
else:
    print("\n有空值，請檢查資料")

# 4. 印出資料筆數
print("\n資料筆數：", len(df))

# 5. 印出資料模型
print("\n資料模型：")

# 接著確定一下我們的欄位
# 方便等一下我們 

df.info()

目前工作目錄：
c:\Users\User\Desktop\機器學習1\model

各欄位空值數量：
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

有空值，請檢查資料

資料筆數： 20640

資料模型：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  floa

In [7]:
df.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='object')

- pandas
   - 讀取data/housing.csv
- scikit learn  做  regression
-  x 的部份  欄位名包含 Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='object')

-  y 的部份  欄位是 ['median_house_value']
- 第一部分先做 train-size-split
   - test-size=0.2
   - random state=123
- ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income']這些欄位要做以下處理
       - 'total_bedrooms'用KNNImputer填補NaN
       - 全部欄位 用minmaxscaler做ˇ轉換
- ' ocean_proximity' 是文字 用one-hot轉換
-  fit後 要顯示MSE 和 r平方的計算結果
- 模型要用pipe組裝  保留preprocess 方便我存檔再利用
- 最後存檔 model/cal_house.pkl
- 給我完整.py檔   不要分段

 

In [13]:
import os
from pathlib import Path

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


# ============================================================
# 1. 找到 housing.csv
# ============================================================

# 先找目前工作目錄下的 data/housing.csv
data_path = Path("data/housing.csv")

# 如果找不到，搜尋常見資料夾
if not data_path.exists():

    search_folders = [
        Path.home() / "Desktop",
        Path.home() / "Documents",
        Path.home() / "Downloads"
    ]

    found_file = None

    for folder in search_folders:

        if folder.exists():

            for file in folder.rglob("housing.csv"):

                found_file = file
                break

        if found_file is not None:
            break

    if found_file is not None:
        data_path = found_file

    else:
        print("找不到 housing.csv")
        print("目前工作目錄：")
        print(os.getcwd())
        raise FileNotFoundError(
            "找不到 housing.csv，請確認 housing.csv 是否存在。"
        )


print("========================================")
print("資料檔案位置")
print("========================================")
print(data_path)


# ============================================================
# 2. 使用 pandas 讀取資料
# ============================================================

df = pd.read_csv(data_path)


print("\n========================================")
print("資料前 5 筆")
print("========================================")

print(df.head())


print("\n========================================")
print("資料欄位")
print("========================================")

print(df.columns)


# ============================================================
# 3. 設定 X 與 y
# ============================================================

X = df.drop(
    columns=["median_house_value"]
)

y = df["median_house_value"]


# ============================================================
# 4. 設定數值欄位
# ============================================================

numeric_features = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income"
]


# ============================================================
# 5. 設定文字欄位
# ============================================================

categorical_features = [
    "ocean_proximity"
]


# ============================================================
# 6. 數值欄位 preprocessing
#
# total_bedrooms 有 NaN
# 使用 KNNImputer
#
# 所有數值欄位
# 使用 MinMaxScaler
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            KNNImputer()
        ),
        (
            "scaler",
            MinMaxScaler()
        )
    ]
)


# ============================================================
# 7. 文字欄位 preprocessing
#
# ocean_proximity
# 使用 One-Hot Encoding
# ============================================================

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# ============================================================
# 8. 建立 preprocess
# ============================================================

preprocess = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)


# ============================================================
# 9. 建立 Regression Pipeline
# ============================================================

model = Pipeline(
    steps=[
        (
            "preprocess",
            preprocess
        ),
        (
            "regression",
            LinearRegression()
        )
    ]
)


# ============================================================
# 10. Train / Test Split
#
# test_size = 0.2
# random_state = 123
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123
)


print("\n========================================")
print("Train / Test Split")
print("========================================")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


# ============================================================
# 11. Fit 模型
# ============================================================

print("\n========================================")
print("開始訓練模型")
print("========================================")

model.fit(
    X_train,
    y_train
)

print("模型訓練完成")


# ============================================================
# 12. Prediction
# ============================================================

y_pred = model.predict(
    X_test
)


# ============================================================
# 13. 計算 MSE
# ============================================================

mse = mean_squared_error(
    y_test,
    y_pred
)


# ============================================================
# 14. 計算 R²
# ============================================================

r2 = r2_score(
    y_test,
    y_pred
)


# ============================================================
# 15. 顯示 MSE 與 R²
# ============================================================

print("\n========================================")
print("Regression Result")
print("========================================")

print(f"MSE = {mse:.2f}")

print(f"R²  = {r2:.4f}")


# ============================================================
# 16. 顯示 preprocess
# ============================================================

print("\n========================================")
print("Preprocess")
print("========================================")

print(model.named_steps["preprocess"])


# ============================================================
# 17. 建立 model 資料夾
# ============================================================

model_folder = Path("model")

model_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 18. 儲存完整 Pipeline
# ============================================================

model_path = model_folder / "cal_house.pkl"

joblib.dump(
    model,
    model_path
)


# ============================================================
# 19. 完成
# ============================================================

print("\n========================================")
print("模型儲存完成")
print("========================================")

print(f"模型位置：{model_path}")

資料檔案位置
C:\Users\User\Desktop\機器學習1\data\housing.csv

資料前 5 筆
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  

資料欄位
Index(['

- 給我load model後 丟一個測試資料做predict的程式碼

In [3]:
import pandas as pd
import joblib
model=joblib.load('model/cal_house.pkl')
data = pd.DataFrame([[
    -122.16,
    37.77,
    47,
    1256,
    2000,
    570,
    218,
    4.375,
    "NEAR BAY"
]], columns=[
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "ocean_proximity"
])
print(model.predict(data))

[435562.72270864]
